# Decode Profiling Notebook

这个 notebook 用来读取 `Decode/WSE-3/run_sim.sh` 生成的 profiling artifact，查看总统计、phase breakdown、category totals 和 PE heatmap。

使用方式：
1. 先运行 `bash run_sim.sh [config.json] [artifact_dir]`
2. 修改下面代码单元里的 `artifact_dir`
3. 依次运行后面的 cells

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("default")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

artifact_dir = Path("profiling_runs/phase_timer_smoke")
artifact_dir

In [ ]:
with open(artifact_dir / "manifest.json", "r", encoding="utf-8") as f:
    manifest = json.load(f)
with open(artifact_dir / "metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)
with open(artifact_dir / "phase_summary.json", "r", encoding="utf-8") as f:
    phase_summary = json.load(f)
with open(artifact_dir / "category_summary.json", "r", encoding="utf-8") as f:
    category_summary = json.load(f)
with open(artifact_dir / "phase_group_summary.json", "r", encoding="utf-8") as f:
    phase_group_summary = json.load(f)

cycles_count = np.load(artifact_dir / "cycles_count.npy")
phase_cycles = np.load(artifact_dir / "phase_cycles.npy")

manifest, metrics

In [ ]:
print("Config:")
for key, value in manifest["config"].items():
    print(f"  {key}: {value}")

print("\nMetrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

print("\nPhase means:")
for key, value in phase_summary.items():
    print(f"  {key}: {value:.3f}")

In [ ]:
category_order = [
    "Compute (Dist-GEMV)",
    "KV-cache GEMV",
    "Communication",
    "Elementwise/Norm",
    "Other",
]
category_colors = {
    "Compute (Dist-GEMV)": "#4E79A7",
    "KV-cache GEMV": "#76B7B2",
    "Communication": "#F28E2B",
    "Elementwise/Norm": "#EDC948",
    "Other": "#9C755F",
}

labels = [entry["label"] for entry in phase_group_summary]
x = np.arange(len(labels))
bottom = np.zeros(len(labels))

fig, ax = plt.subplots(figsize=(12, 5))
for category in category_order:
    values = np.array([entry["segments"].get(category, 0.0) for entry in phase_group_summary], dtype=float)
    ax.bar(x, values, bottom=bottom, label=category, color=category_colors[category])
    bottom += values

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_ylabel("Cycles")
ax.set_title("Decode Phase Breakdown")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cats = list(category_summary.keys())
vals = [category_summary[c] for c in cats]
colors = [category_colors.get(c, "#888888") for c in cats]
ax.bar(cats, vals, color=colors)
ax.set_ylabel("Cycles")
ax.set_title("Decode Category Totals")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cycles_count, cmap="viridis")
ax.set_title("Decode Cycles Heatmap")
ax.set_xlabel("PE x")
ax.set_ylabel("PE y")
fig.colorbar(im, ax=ax, label="Cycles")
plt.tight_layout()
plt.show()